# CloudSentinel AI
## Intelligent Cloud Resource Monitoring & Cost Anomaly Detection

### Notebook 01 — Data Generation and Exploration

This notebook creates the synthetic cloud-resource dataset used by
the other CloudSentinel AI notebooks. Run it only when you want to
regenerate the CSV; the current dataset already exists in `data/raw`.

## Run guidance

To regenerate the raw dataset, run every code cell from top to bottom. This overwrites `data/raw/cloud_resource_metrics.csv`. The CSV is already present, so this notebook is optional for exploration and detection.

## 1. Import Required Libraries

In [ ]:
# Core Python libraries
from pathlib import Path
import random

# Data manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Reproducibility
RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

print("Libraries imported successfully.")

## 2. Project Configuration

In [ ]:
# Project root directory
PROJECT_ROOT = Path.cwd().parent

# Data directories
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

# Model directory
MODEL_DIR = PROJECT_ROOT / "models"

# Create directories if they do not already exist
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data directory: {RAW_DATA_DIR}")
print(f"Processed data directory: {PROCESSED_DATA_DIR}")
print(f"Model directory: {MODEL_DIR}")

## 3. Define Simulated Cloud Resources

We simulate a small enterprise environment containing different
types of cloud resources.

Each resource has different baseline utilization characteristics
to make the synthetic dataset more realistic.

In [ ]:
# Simulated cloud resources
resources = [
    {
        "resource_id": "EC2-001",
        "resource_type": "Web Server",
        "base_cpu": 35,
        "base_memory": 50,
        "base_network": 120,
        "base_requests": 1500,
        "base_storage": 55
    },
    {
        "resource_id": "EC2-002",
        "resource_type": "Web Server",
        "base_cpu": 40,
        "base_memory": 52,
        "base_network": 140,
        "base_requests": 1800,
        "base_storage": 58
    },
    {
        "resource_id": "EC2-003",
        "resource_type": "API Server",
        "base_cpu": 45,
        "base_memory": 58,
        "base_network": 180,
        "base_requests": 2500,
        "base_storage": 62
    },
    {
        "resource_id": "EC2-004",
        "resource_type": "Database Server",
        "base_cpu": 50,
        "base_memory": 68,
        "base_network": 100,
        "base_requests": 1200,
        "base_storage": 72
    },
    {
        "resource_id": "EC2-005",
        "resource_type": "Worker Server",
        "base_cpu": 30,
        "base_memory": 45,
        "base_network": 90,
        "base_requests": 800,
        "base_storage": 48
    }
]

resources_df = pd.DataFrame(resources)

resources_df

## 4. Define Monitoring Period

CloudSentinel collects resource metrics at 15-minute intervals
over a 30-day simulated monitoring period.

In [ ]:
# Simulation configuration
START_DATE = "2026-08-01"
DAYS = 30
FREQUENCY = "15min"

timestamps = pd.date_range(
    start=START_DATE,
    periods=DAYS * 24 * 4,
    freq=FREQUENCY
)

print(f"Monitoring start: {timestamps.min()}")
print(f"Monitoring end:   {timestamps.max()}")
print(f"Total timestamps:  {len(timestamps)}")

## 5. Define Normal Workload Patterns

Cloud resources typically experience different utilization levels
throughout the day.

We model this behavior so that the anomaly detector does not
mistake normal daily peaks for abnormal events.

In [ ]:
def workload_multiplier(hour):
    """
    Returns a realistic workload multiplier based on time of day.
    """

    if 0 <= hour < 6:
        return 0.55          # Low overnight activity

    elif 6 <= hour < 9:
        return 0.75          # Morning increase

    elif 9 <= hour < 13:
        return 1.00          # Normal working hours

    elif 13 <= hour < 18:
        return 1.15          # Afternoon peak

    elif 18 <= hour < 22:
        return 1.05          # Evening activity

    else:
        return 0.70          # Late night


# Test the function
for hour in [2, 8, 11, 15, 20, 23]:
    print(f"{hour:02d}:00 → {workload_multiplier(hour)}")

## 6. Generate Normal Metrics for One Resource

Before generating the complete dataset, we test the simulation
logic using a single cloud resource.

In [ ]:
# Select one resource for testing
test_resource = resources[0]

test_data = []

for timestamp in timestamps:

    # Time-based workload
    multiplier = workload_multiplier(timestamp.hour)

    # Add small random variation around the baseline
    cpu = (
        test_resource["base_cpu"]
        * multiplier
        + np.random.normal(0, 3)
    )

    memory = (
        test_resource["base_memory"]
        * multiplier
        + np.random.normal(0, 2)
    )

    network = (
        test_resource["base_network"]
        * multiplier
        + np.random.normal(0, 10)
    )

    requests = (
        test_resource["base_requests"]
        * multiplier
        + np.random.normal(0, 100)
    )

    storage = (
        test_resource["base_storage"]
        + np.random.normal(0, 1)
    )

    # Keep values within realistic boundaries
    cpu = np.clip(cpu, 0, 100)
    memory = np.clip(memory, 0, 100)
    storage = np.clip(storage, 0, 100)
    network = max(network, 0)
    requests = max(requests, 0)

    test_data.append({
        "timestamp": timestamp,
        "resource_id": test_resource["resource_id"],
        "resource_type": test_resource["resource_type"],
        "cpu_utilization": cpu,
        "memory_utilization": memory,
        "network_in": network,
        "request_count": requests,
        "storage_usage": storage
    })

test_df = pd.DataFrame(test_data)

test_df.head()

## 7. Visualize Normal CPU Behavior

We visualize CPU utilization over time to verify that the
simulated workload follows realistic daily patterns.

In [ ]:
plt.figure(figsize=(14, 5))

plt.plot(
    test_df["timestamp"],
    test_df["cpu_utilization"]
)

plt.title("Simulated CPU Utilization — EC2-001")
plt.xlabel("Time")
plt.ylabel("CPU Utilization (%)")

plt.xticks(rotation=45)
plt.tight_layout()

plt.show()

## 8. Estimate Resource Cost

For the synthetic environment, we create an estimated cost metric
based on resource utilization.

This is a simulation for ML development and does not represent
actual AWS billing.

In [ ]:
# Calculate a synthetic estimated cost for the test resource

test_df["estimated_cost"] = (
    0.04
    + (test_df["cpu_utilization"] / 100) * 0.04
    + (test_df["memory_utilization"] / 100) * 0.03
    + (test_df["network_in"] / 1000) * 0.01
    + (test_df["request_count"] / 5000) * 0.02
)

test_df["estimated_cost"] = test_df["estimated_cost"].round(4)

test_df.head()

## 9. Generate the Complete Cloud Resource Dataset

We now extend the single-resource simulation to all simulated
enterprise resources.

In [ ]:
all_data = []

for resource in resources:

    for timestamp in timestamps:

        # Normal workload based on time of day
        multiplier = workload_multiplier(timestamp.hour)

        cpu = (
            resource["base_cpu"] * multiplier
            + np.random.normal(0, 3)
        )

        memory = (
            resource["base_memory"] * multiplier
            + np.random.normal(0, 2)
        )

        network = (
            resource["base_network"] * multiplier
            + np.random.normal(0, 10)
        )

        requests = (
            resource["base_requests"] * multiplier
            + np.random.normal(0, 100)
        )

        storage = (
            resource["base_storage"]
            + np.random.normal(0, 1)
        )

        # Keep values realistic
        cpu = np.clip(cpu, 0, 100)
        memory = np.clip(memory, 0, 100)
        storage = np.clip(storage, 0, 100)

        network = max(network, 0)
        requests = max(requests, 0)

        # Synthetic cost model
        estimated_cost = (
            0.04
            + (cpu / 100) * 0.04
            + (memory / 100) * 0.03
            + (network / 1000) * 0.01
            + (requests / 5000) * 0.02
        )

        all_data.append({
            "timestamp": timestamp,
            "resource_id": resource["resource_id"],
            "resource_type": resource["resource_type"],
            "cpu_utilization": cpu,
            "memory_utilization": memory,
            "network_in": network,
            "request_count": requests,
            "storage_usage": storage,
            "estimated_cost": estimated_cost,
            "is_anomaly": 0,
            "anomaly_type": "normal"
        })

cloud_df = pd.DataFrame(all_data)

print(f"Rows: {len(cloud_df):,}")
print(f"Columns: {len(cloud_df.columns)}")

In [ ]:
cloud_df.head()

In [ ]:
cloud_df.info()

In [ ]:
cloud_df.describe()

## 10. Inject CPU Spike Anomalies

We intentionally introduce short periods of abnormally high CPU
utilization to simulate sudden workload spikes.

In [ ]:
# Select a deterministic set of rows for CPU anomalies
cpu_anomaly_indices = cloud_df.sample(
    n=25,
    random_state=RANDOM_STATE
).index

cloud_df.loc[cpu_anomaly_indices, "cpu_utilization"] = np.random.uniform(
    90, 99, size=len(cpu_anomaly_indices)
)

cloud_df.loc[cpu_anomaly_indices, "is_anomaly"] = 1
cloud_df.loc[cpu_anomaly_indices, "anomaly_type"] = "cpu_spike"

print(f"CPU anomalies injected: {len(cpu_anomaly_indices)}")

## 11. Inject Memory Spike Anomalies

In [ ]:
memory_anomaly_indices = cloud_df[
    cloud_df["is_anomaly"] == 0
].sample(
    n=20,
    random_state=RANDOM_STATE + 1
).index

cloud_df.loc[memory_anomaly_indices, "memory_utilization"] = np.random.uniform(
    90, 98, size=len(memory_anomaly_indices)
)

cloud_df.loc[memory_anomaly_indices, "is_anomaly"] = 1
cloud_df.loc[memory_anomaly_indices, "anomaly_type"] = "memory_spike"

print(f"Memory anomalies injected: {len(memory_anomaly_indices)}")

## 12. Inject Network Traffic Anomalies

In [ ]:
network_anomaly_indices = cloud_df[
    cloud_df["is_anomaly"] == 0
].sample(
    n=20,
    random_state=RANDOM_STATE + 2
).index

cloud_df.loc[network_anomaly_indices, "network_in"] = np.random.uniform(
    800, 1200, size=len(network_anomaly_indices)
)

cloud_df.loc[network_anomaly_indices, "is_anomaly"] = 1
cloud_df.loc[network_anomaly_indices, "anomaly_type"] = "network_spike"

print(f"Network anomalies injected: {len(network_anomaly_indices)}")

## 13. Inject Cost Anomalies

We simulate situations where estimated resource cost increases
unexpectedly relative to normal operating behavior.

In [ ]:
cost_anomaly_indices = cloud_df[
    cloud_df["is_anomaly"] == 0
].sample(
    n=20,
    random_state=RANDOM_STATE + 3
).index

cloud_df.loc[cost_anomaly_indices, "estimated_cost"] *= np.random.uniform(
    2.5, 4.0, size=len(cost_anomaly_indices)
)

cloud_df.loc[cost_anomaly_indices, "is_anomaly"] = 1
cloud_df.loc[cost_anomaly_indices, "anomaly_type"] = "cost_spike"

print(f"Cost anomalies injected: {len(cost_anomaly_indices)}")

## 14. Inject Combined Resource Anomalies

A combined anomaly represents a significant workload event where
multiple resource metrics increase simultaneously.

In [ ]:
combined_indices = cloud_df[
    cloud_df["is_anomaly"] == 0
].sample(
    n=15,
    random_state=RANDOM_STATE + 4
).index

cloud_df.loc[combined_indices, "cpu_utilization"] = np.random.uniform(
    85, 98, size=len(combined_indices)
)

cloud_df.loc[combined_indices, "memory_utilization"] = np.random.uniform(
    80, 95, size=len(combined_indices)
)

cloud_df.loc[combined_indices, "network_in"] = np.random.uniform(
    700, 1100, size=len(combined_indices)
)

cloud_df.loc[combined_indices, "request_count"] *= np.random.uniform(
    2.0, 3.0, size=len(combined_indices)
)

cloud_df.loc[combined_indices, "estimated_cost"] *= np.random.uniform(
    2.0, 3.0, size=len(combined_indices)
)

cloud_df.loc[combined_indices, "is_anomaly"] = 1
cloud_df.loc[combined_indices, "anomaly_type"] = "combined_spike"

print(f"Combined anomalies injected: {len(combined_indices)}")

## 15. Inspect Anomaly Distribution

In [ ]:
cloud_df["anomaly_type"].value_counts()

In [ ]:
anomaly_percentage = (
    cloud_df["is_anomaly"].mean() * 100
)

print(f"Total observations: {len(cloud_df):,}")
print(f"Anomalous observations: {cloud_df['is_anomaly'].sum():,}")
print(f"Anomaly percentage: {anomaly_percentage:.2f}%")

## 16. Save Raw Synthetic Dataset

The generated dataset is saved as the raw input for subsequent
CloudSentinel ML experiments.

In [ ]:
RAW_DATA_FILE = RAW_DATA_DIR / "cloud_resource_metrics.csv"

cloud_df.to_csv(
    RAW_DATA_FILE,
    index=False
)

print(f"Dataset saved to: {RAW_DATA_FILE}")
print(f"Dataset shape: {cloud_df.shape}")

In [ ]:
cloud_df.sample(
    10,
    random_state=RANDOM_STATE
)

In [ ]:
cloud_df.isnull().sum()